In [1]:
import sys
# sys.path.insert(0, r"C:\SVN\geosaurus_master\src")
from datetime import datetime
from arcgis.features import FeatureCollection, FeatureSet
from arcgis.gis import Item
from arcgis.geoprocessing._job import GPJob

In [2]:
from arcgis.auth.tools._util import detect_proxy

In [3]:
from arcgis.gis import GIS
#gis = GIS(username="api_data_owner", password="data.owner8")
gis = GIS('https://deldev.maps.arcgis.com', 'demos_deldev', 'DelDevs.1234', verify_cert=False, proxy=detect_proxy(True))

In [4]:
origin_item = gis.content.get('1f163950afbb4bf39f3ee67cf411761d')
dest_item = gis.content.get('c7665d3c8e6f48a79f07b79677996bed')
point_barrier = gis.content.get('d692355520e94d39a028f79248b75ef7')

In [5]:
item =  gis.content.search('Traffic Collisions owner: api_data_owner', 'feature layer', outside_org=True)

In [6]:
item

[<Item title:"Traffic Collisions" type:Feature Layer Collection owner:api_data_owner>]

In [7]:
#item[0].layers[0].query(as_df=True).columns

In [8]:
from arcgis.features.analyze_patterns import find_outliers

In [9]:
test = find_outliers(analysis_layer=item[0].layers[0], shape_type='fishnet')

In [10]:
assert isinstance(test, dict)

In [11]:
test

{'outliers_result_layer': <FeatureCollection>,
 'process_info': ['{"messageCode": "SS_00004", "message": "The following report outlines the workflow used to optimize your Find Outliers result:", "params": {}, "style": "<b></b><br/>"}',
  '{"message": "Initial Data Assessment", "messageCode": "SS_84428", "params": {}, "style": "<u><b></b></u><br/>"}',
  '{"message": "There are ${NumFeatures} valid input features.", "messageCode": "SS_84485", "params": {"NumFeatures": "17031"}, "style": "<ul><li></li></ul>"}',
  '{"message": "There were {numOutliers} outlier locations; these will not be used to compute the polygon cell size.", "messageCode": "SS_84495_1", "params": {"numOutliers": "130"}, "style": "<ul><li></li></ul>"}',
  '{"message": "Incident Aggregation", "messageCode": "SS_84444", "params": {}, "style": "<u><b></b></u><br/>"}',
  '{"message": "Using a polygon cell size of ${SnapInfo}", "messageCode": "SS_84450", "params": {"SnapInfo": "118.0000 Meters"}, "style": "<ul><li></li></ul>

In [12]:
assert isinstance(test['outliers_result_layer'], FeatureCollection)

In [13]:
job = find_outliers(analysis_layer=item[0].layers[0], shape_type='fishnet', future=True)
assert isinstance(job, GPJob)

In [14]:
assert isinstance(job.result()['outliers_result_layer'], FeatureCollection)

In [15]:
assert isinstance(job.result(), dict)

#### Test Overwrite

In [16]:
#import sys
#sys.path.insert(0, r"C:\\ipython_workfolder\\geosaurus\\src")
try:
    from arcgis.gis import GIS, ProfileManager
    pm = ProfileManager().list()
    if 'your_enterprise_profile' in pm:
        portal = GIS(profile="your_enterprise_profile", verify_cert=False,)
        item = portal.content.get("7a1ecc59e0a443d485dfd123b7a293ef")
        if item:
            test = find_outliers(
                analysis_layer=item.layers[0],
                shape_type="fishnet",
                output_name="test_outliers_overwrite",)
            fl_to_overwrite = portal.content.get(test["outliers_result_layer"].id).layers[0]
            overwrite = find_outliers(analysis_layer=item.layers[0],shape_type="hexagon",
                    output_name=fl_to_overwrite,
                    context={"overwrite": True})
            # Notice the ids are the same, if overwrite fails then fl is appended instead
            print(overwrite["outliers_result_layer"].layers)
            print(overwrite["outliers_result_layer"].id)
            overwrite["outliers_result_layer"].delete()
except Exception as e:
    print(e)